# NB_patient_360

Builds `bharath_dev_adb_p1.gold.patient_360` (one row per patient) from the silver layer per the STTM (Silver to Gold, v1.0). Driving table `silver.patients`, LEFT JOINed to hospitals, insurance_providers and per-patient lab_results aggregates. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE bharath_dev_adb_p1.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (STTM Silver to Gold v1.0)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM bharath_dev_adb_p1.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                              AS date_of_birth,  -- silver stores TIMESTAMP
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                  AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                                          AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                   AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                           AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                           AS critical_test_count,
  l.last_test_date
FROM bharath_dev_adb_p1.silver.patients p
LEFT JOIN bharath_dev_adb_p1.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN bharath_dev_adb_p1.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
# Return the gold table record count to the caller (e.g. a job or dbutils.notebook.run)
gold_df = spark.table("bharath_dev_adb_p1.gold.patient_360")

record_count = gold_df.count()
print(f"[STEP 5] Exiting notebook with processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))